In [1]:
import pandas as pd
import matplotlib.pyplot as plt

from datetime import datetime
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils import resample

In [2]:
train = pd.read_csv("train.csv")

In [3]:
# datetime列をdatetime64型に変換
train["datetime"] = pd.to_datetime(train["datetime"])

# year, month, day, weekday列を作成
train["year"] = train["datetime"].dt.year
train["month"] = train["datetime"].dt.month
train["day"] = train["datetime"].dt.day
train["weekday"] = train["datetime"].dt.dayofweek

# 不要なdatetime列を削除
train.drop("datetime", axis=1, inplace=True)

train.head()

,y,client,close,price_am,price_pm,year,month,day,weekday
0,17,0,0,-1,-1,2010,7,1,3
1,18,0,0,-1,-1,2010,7,2,4
2,20,0,0,-1,-1,2010,7,3,5
3,20,0,0,-1,-1,2010,7,4,6
4,14,0,0,-1,-1,2010,7,5,0


In [4]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2101 entries, 0 to 2100
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   y         2101 non-null   int64
 1   client    2101 non-null   int64
 2   close     2101 non-null   int64
 3   price_am  2101 non-null   int64
 4   price_pm  2101 non-null   int64
 5   year      2101 non-null   int32
 6   month     2101 non-null   int32
 7   day       2101 non-null   int32
 8   weekday   2101 non-null   int32
dtypes: int32(4), int64(5)
memory usage: 115.0 KB


In [5]:
# 休日フラグがオンでもyが0以外なら休日フラグをオフにする
#train.loc[(train["close"] == 1) & (train["y"] != 0), "close"] = 0

#休日フラグがオンなら無条件で0にする
#train.loc[train["close"] == 1, ["price_am", "price_pm"]] = 0

#休日フラグがオンは除外(最後に強制的に0にする)
train = train.query("close != 1")

In [6]:
#price_am,pmの欠損とそれ以外でtrain,testに分割
train_am = train.query("price_am != -1")
train_am = train_am.drop(columns=["price_pm"])
test_am = train.query("price_am == -1")
test_am = test_am.drop(columns=["price_am", "price_pm"])

train_pm = train.query("price_pm != -1")
train_pm =train_pm.drop(columns=["price_am"])
test_pm = train.query("price_pm == -1")
test_pm = test_pm.drop(columns=["price_am", "price_pm"])

In [7]:
#priceをそれぞれに分割
train_am_Zero = train_am.query("price_am == 0")
train_am_One = train_am.query("price_am == 1")
train_am_Two = train_am.query("price_am == 2")
train_am_Three = train_am.query("price_am == 3")
train_am_Four = train_am.query("price_am == 4")
train_am_Five = train_am.query("price_am == 5")

train_pm_Zero = train_pm.query("price_pm == 0")
train_pm_One = train_pm.query("price_pm == 1")
train_pm_Two = train_pm.query("price_pm == 2")
train_pm_Three = train_pm.query("price_pm == 3")
train_pm_Four = train_pm.query("price_pm == 4")
train_pm_Five = train_pm.query("price_pm == 5")

In [8]:
#0に対してアンダーサンプリング
downsampled_train_am_Zero = resample(train_am_Zero, replace=False, n_samples=500, random_state=42)

downsampled_train_pm_Zero = resample(train_pm_Zero, replace=False, n_samples=500, random_state=42)

In [9]:
#0以外に対してオーバーサンプリング
upsampled_train_am_One = resample(train_am_One, replace=True, n_samples=700, random_state=42)
upsampled_train_am_Two = resample(train_am_Two, replace=True, n_samples=700, random_state=42)
upsampled_train_am_Three = resample(train_am_Three, replace=True, n_samples=700, random_state=42)
upsampled_train_am_Four = resample(train_am_Four, replace=True, n_samples=700, random_state=42)
upsampled_train_am_Five = resample(train_am_Five, replace=True, n_samples=700, random_state=42)

upsampled_train_pm_One = resample(train_pm_One, replace=True, n_samples=700, random_state=42)
upsampled_train_pm_Two = resample(train_pm_Two, replace=True, n_samples=700, random_state=42)
upsampled_train_pm_Three = resample(train_pm_Three, replace=True, n_samples=700, random_state=42)
upsampled_train_pm_Four = resample(train_pm_Four, replace=True, n_samples=700, random_state=42)
upsampled_train_pm_Five = resample(train_pm_Five, replace=True, n_samples=700, random_state=42)

In [10]:
#マージ
balanced_train_am = pd.concat([downsampled_train_am_Zero, upsampled_train_am_One, upsampled_train_am_Two, upsampled_train_am_Three, upsampled_train_am_Four, upsampled_train_am_Five])

balanced_train_pm = pd.concat([downsampled_train_pm_Zero, upsampled_train_pm_One, upsampled_train_pm_Two, upsampled_train_pm_Three, upsampled_train_pm_Four, upsampled_train_pm_Five])

In [11]:
#Xとyに分割
X_am = balanced_train_am.drop(columns=["price_am"])
y_am = balanced_train_am["price_am"]

X_pm = balanced_train_pm.drop(columns=["price_pm"])
y_pm = balanced_train_pm["price_pm"]

In [12]:
#最適化及びモデル作成
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

model = RandomForestClassifier()

grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search_am = grid_search.fit(X_am, y_am)
grid_search_pm = grid_search.fit(X_pm, y_pm)

best_params_am = grid_search_am.best_params_
print("Best Parameters am:", best_params_am)
best_params_pm = grid_search_pm.best_params_
print("Best Parameters pm :", best_params_pm)

best_model_am = grid_search_am.best_estimator_
best_model_pm = grid_search_pm.best_estimator_

Best Parameters am: {'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100}
Best Parameters pm : {'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100}


In [13]:
#評価
cv_scores_am = cross_val_score(grid_search_am.best_estimator_, X_am, y_am, cv=5, scoring='accuracy')
cv_scores_pm = cross_val_score(grid_search_pm.best_estimator_, X_pm, y_pm, cv=5, scoring='accuracy')

In [14]:
#評価の表示
print("Cross Validation Scores am:", cv_scores_am)
print("Cross Validation Scores pm:", cv_scores_pm)

Cross Validation Scores am: [0.93875 0.94375 0.94    0.94375 0.9275 ]
Cross Validation Scores pm: [0.975   0.96    0.95875 0.96375 0.9725 ]


In [15]:
#評価平均値の表示
print("Mean Score am:", cv_scores_am.mean())
print("Mean Score pm:", cv_scores_pm.mean())

Mean Score am: 0.93875
Mean Score pm: 0.966


In [16]:
#testの予測
y_pred_am = best_model_am.predict(test_am)

y_pred_pm = best_model_pm.predict(test_pm)

In [17]:
#price_を戻す
test_am["price_am"] = y_pred_am

test_pm["price_pm"] = y_pred_pm

In [18]:
#マージしてソートして同じインデックスは統合
pre_data = pd.concat([train_am, test_am, train_pm, test_pm])

pre_data = pre_data.sort_index()

pre_data = pre_data.groupby(pre_data.index).first()

In [19]:
#train = pd.read_csv("train.csv")

# datetime列をdatetime64型に変換
#train["datetime"] = pd.to_datetime(train["datetime"])

# 日付の範囲を決定
#min_date = train["datetime"].min()
#max_date = train["datetime"].max()

# 日付を0から1の範囲にスケーリング
#pre_data["datetime"] = (train["datetime"] - min_date) / (max_date - min_date)

pre_data.head()

,y,client,close,price_am,year,month,day,weekday,price_pm
0,17,0,0,0.0,2010,7,1,3,0.0
1,18,0,0,0.0,2010,7,2,4,0.0
2,20,0,0,1.0,2010,7,3,5,1.0
3,20,0,0,1.0,2010,7,4,6,1.0
4,14,0,0,0.0,2010,7,5,0,0.0


In [20]:
#休日フラグオンを分割
#close_data = pre_data.query("close == 1")

#clientが1とそれ以外に分割
client_data = pre_data.query("client == 1")
pre_data = pre_data.query("client == 0")

In [21]:
#clientを分割
client_am_Zero = client_data.query("price_am == 0")
client_am_One = client_data.query("price_am == 1")
client_am_Two = client_data.query("price_am == 2")
client_am_Three = client_data.query("price_am == 3")
client_am_Four = client_data.query("price_am == 4")
client_am_Five = client_data.query("price_am == 5")

client_pm_Zero = client_data.query("price_pm == 0")
client_pm_One = client_data.query("price_pm == 1")
client_pm_Two = client_data.query("price_pm == 2")
client_pm_Three = client_data.query("price_pm == 3")
client_pm_Four = client_data.query("price_pm == 4")
client_pm_Five = client_data.query("price_pm == 5")

#priceをそれぞれに分割
pre_am_Zero = pre_data.query("price_am == 0")
pre_am_One = pre_data.query("price_am == 1")
pre_am_Two = pre_data.query("price_am == 2")
pre_am_Three = pre_data.query("price_am == 3")
pre_am_Four = pre_data.query("price_am == 4")
pre_am_Five = pre_data.query("price_am == 5")

pre_pm_Zero = pre_data.query("price_pm == 0")
pre_pm_One = pre_data.query("price_pm == 1")
pre_pm_Two = pre_data.query("price_pm == 2")
pre_pm_Three = pre_data.query("price_pm == 3")
pre_pm_Four = pre_data.query("price_pm == 4")
pre_pm_Five = pre_data.query("price_pm == 5")

In [22]:
#オーバーサンプリング
upsampled_client_am_Zero = resample(client_am_Zero, replace=True, n_samples=700, random_state=42)
upsampled_client_am_One = resample(client_am_One, replace=True, n_samples=700, random_state=42)
upsampled_client_am_Two = resample(client_am_Two, replace=True, n_samples=700, random_state=42)
upsampled_client_am_Three = resample(client_am_Three, replace=True, n_samples=700, random_state=42)
upsampled_client_am_Four = resample(client_am_Four, replace=True, n_samples=700, random_state=42)
upsampled_client_am_Five = resample(client_am_Five, replace=True, n_samples=700, random_state=42)

upsampled_client_pm_Zero = resample(client_pm_Zero, replace=True, n_samples=700, random_state=42)
upsampled_client_pm_One = resample(client_pm_One, replace=True, n_samples=700, random_state=42)
upsampled_client_pm_Two = resample(client_pm_Two, replace=True, n_samples=700, random_state=42)
upsampled_client_pm_Three = resample(client_pm_Three, replace=True, n_samples=700, random_state=42)
upsampled_client_pm_Four = resample(client_pm_Four, replace=True, n_samples=700, random_state=42)
upsampled_client_pm_Five = resample(client_pm_Five, replace=True, n_samples=700, random_state=42)

In [23]:
#0に対してアンダーサンプリング
downsampled_pre_am_Zero = resample(pre_am_Zero, replace=False, n_samples=500, random_state=42)

downsampled_pre_pm_Zero = resample(pre_pm_Zero, replace=False, n_samples=500, random_state=42)

In [24]:
#0以外に対してオーバーサンプリング
upsampled_pre_am_One = resample(pre_am_One, replace=True, n_samples=700, random_state=42)
upsampled_pre_am_Two = resample(pre_am_Two, replace=True, n_samples=700, random_state=42)
upsampled_pre_am_Three = resample(pre_am_Three, replace=True, n_samples=700, random_state=42)
upsampled_pre_am_Four = resample(pre_am_Four, replace=True, n_samples=700, random_state=42)
upsampled_pre_am_Five = resample(pre_am_Five, replace=True, n_samples=700, random_state=42)

upsampled_pre_pm_One = resample(pre_pm_One, replace=True, n_samples=700, random_state=42)
upsampled_pre_pm_Two = resample(pre_pm_Two, replace=True, n_samples=700, random_state=42)
upsampled_pre_pm_Three = resample(pre_pm_Three, replace=True, n_samples=700, random_state=42)
upsampled_pre_pm_Four = resample(pre_pm_Four, replace=True, n_samples=700, random_state=42)
upsampled_pre_pm_Five = resample(pre_pm_Five, replace=True, n_samples=700, random_state=42)

In [25]:
#マージ
balanced_client_am = pd.concat([upsampled_client_am_Zero, upsampled_client_am_One, upsampled_client_am_Two, upsampled_client_am_Three, upsampled_client_am_Four, upsampled_client_am_Five])

balanced_client_pm = pd.concat([upsampled_client_pm_Zero, upsampled_client_pm_One, upsampled_client_pm_Two, upsampled_client_pm_Three, upsampled_client_pm_Four, upsampled_client_pm_Five])

balanced_client_data = pd.concat([balanced_client_am, balanced_client_pm])

balanced_pre_am = pd.concat([downsampled_pre_am_Zero, upsampled_pre_am_One, upsampled_pre_am_Two, upsampled_pre_am_Three, upsampled_pre_am_Four, upsampled_pre_am_Five])

balanced_pre_pm = pd.concat([downsampled_pre_pm_Zero, upsampled_pre_pm_One, upsampled_pre_pm_Two, upsampled_pre_pm_Three, upsampled_pre_pm_Four, upsampled_pre_pm_Five])

balanced_pre_data = pd.concat([balanced_pre_am, balanced_pre_pm])

In [26]:
date_client = balanced_client_data.drop(columns=["y", "client", "close", "price_am", "price_pm"])

date_pre = balanced_pre_data.drop(columns=["y", "client", "close", "price_am", "price_pm"])

balanced_client_data = balanced_client_data.drop(columns=["year", "month", "day", "weekday"])

balanced_pre_data = balanced_pre_data.drop(columns=["year", "month", "day", "weekday"])

balanced_client_data = pd.concat([balanced_client_data, date_client], axis=1)

balanced_pre_data = pd.concat([balanced_pre_data, date_pre], axis=1)

In [27]:
balanced_client_data.head()

,y,client,close,price_am,price_pm,year,month,day,weekday
1986,29,1,0,0.0,0.0,2015,12,8,1
1631,35,1,0,0.0,0.0,2014,12,18,3
2038,68,1,0,0.0,0.0,2016,1,29,4
1999,65,1,0,0.0,0.0,2015,12,21,0
1658,29,1,0,0.0,0.0,2015,1,14,2


In [28]:
balanced_pre_data.head()

,y,client,close,price_am,price_pm,year,month,day,weekday
378,16,0,0,0.0,0.0,2011,7,14,3
1050,29,0,0,0.0,0.0,2013,5,16,3
329,18,0,0,0.0,0.0,2011,5,26,3
894,22,0,0,0.0,0.0,2012,12,11,1
110,6,0,0,0.0,0.0,2010,10,19,1


In [29]:
#csvへ書き出し
balanced_client_data.to_csv("client_data.csv", index=False)
balanced_pre_data.to_csv("pre_data.csv", index=False)